# 2026-05-18: 1-turn AI per-deck 重み NN を 16 deck loop 学習

OPTCG は リーダー効果で プレイ思想が 根本的に違うので、 archetype 集約より **deck 個別**で 学習する (= ohtsuki さん指摘)。

1 つの Colab セッションで:
1. `snapshots_oneturn.jsonl` 全体 (= 14.4 万 snap) load
2. in-memory で deck slug 別 split (= 16 deck × ~9000 snap)
3. 各 deck で REINFORCE 学習 → `weight_nn_oneturn_<slug>.pt` × 16 出力

ohtsuki さん:
- snapshot upload は `snapshots_oneturn.jsonl` (= 220 MB) のみ
- 完了後 16 個の `.pt` を ローカル `db/nn_per_deck_oneturn/` へ download

学習時間目安: T4 GPU で **全 16 NN 合計 30-45 分** (= 各 ~2 分)

## 1. GPU 確認 + Drive マウント

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/onepiece_nn'
SNAPSHOT_PATH = os.path.join(WORK_DIR, 'snapshots_oneturn.jsonl')
OUTPUT_DIR = os.path.join(WORK_DIR, 'nn_per_deck_oneturn')
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(SNAPSHOT_PATH), f'snapshot 不在: {SNAPSHOT_PATH}'
print(f'snapshot: {SNAPSHOT_PATH} ({os.path.getsize(SNAPSHOT_PATH) // (1024*1024)} MB)')
print(f'output: {OUTPUT_DIR}/')

## 2. snapshot 読み込み + deck slug 別 split

In [ ]:
import json, time
import numpy as np
from collections import defaultdict

STATE_DIM = 172
GAMMA = 0.95

# 各 deck の snapshot を 集約 (= actor が プレイ中の deck)
per_deck = defaultdict(list)
t0 = time.time()
with open(SNAPSHOT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            snap = json.loads(line)
        except json.JSONDecodeError:
            continue
        if 'state_encoded' not in snap or 'reward' not in snap:
            continue
        actor = snap.get('actor_idx', 0)
        deck_slug = snap.get('deck_a') if actor == 0 else snap.get('deck_b')
        if not deck_slug:
            continue
        # discount reward
        delta = max(0, snap.get('max_turn', snap.get('turn', 0)) - snap.get('turn', 0))
        snap['discounted_reward'] = snap['reward'] * (GAMMA ** delta)
        per_deck[deck_slug].append(snap)

print(f'load + split: {time.time()-t0:.1f}s')
print(f'decks: {len(per_deck)}')
for slug in sorted(per_deck.keys(), key=lambda s: -len(per_deck[s])):
    print(f'  {slug:35s}: {len(per_deck[slug]):6d} snaps')

## 3. WeightNN 定義

In [ ]:
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_WEIGHTS = 9
BASE_SCALES = torch.tensor([3000, 800, 1500, 1, 500, 1200, 1000, 600, 30000],
                            dtype=torch.float32, device=device)

class WeightNN(nn.Module):
    def __init__(self, input_dim=172, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.ReLU(),
        )
        self.weight_head = nn.Linear(hidden_dim // 4, N_WEIGHTS)
        self.softplus = nn.Softplus()
        self.register_buffer('base_scales', BASE_SCALES.cpu().clone())
    def forward(self, x):
        h = self.shared(x)
        raw = self.weight_head(h)
        return self.softplus(raw) * self.base_scales

## 4. 16 deck loop 学習

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

EPOCHS = 30
BATCH = 512
LR = 1e-4
NOISE_STD = 0.3

summary = {}
t_global = time.time()

for slug_idx, slug in enumerate(sorted(per_deck.keys())):
    snaps = per_deck[slug]
    if len(snaps) < 100:
        print(f'[{slug_idx+1}/{len(per_deck)}] {slug}: 学習データ不足 ({len(snaps)})、 SKIP')
        continue

    t_deck = time.time()
    print(f'[{slug_idx+1}/{len(per_deck)}] {slug}: {len(snaps)} snaps')

    # tensor 化
    X_np = np.zeros((len(snaps), STATE_DIM), dtype=np.float32)
    R_np = np.zeros((len(snaps),), dtype=np.float32)
    for i, snap in enumerate(snaps):
        se = snap['state_encoded']
        if len(se) >= STATE_DIM:
            X_np[i, :] = se[:STATE_DIM]
        else:
            X_np[i, :len(se)] = se
        R_np[i] = snap['discounted_reward']
    X = torch.from_numpy(X_np).to(device)
    R = torch.from_numpy(R_np).to(device)

    baseline = R.mean()
    advantage = R - baseline

    train_ds = TensorDataset(X, advantage)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)

    model = WeightNN().to(device)
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    final_loss = None
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        n_batches = 0
        for xb, advb in train_loader:
            optimizer.zero_grad()
            mu = model(xb)
            eps = torch.randn_like(mu) * NOISE_STD * mu.abs().mean(dim=1, keepdim=True)
            sampled = mu + eps
            std = NOISE_STD * mu.abs().mean(dim=1, keepdim=True).clamp(min=1.0)
            log_prob = -0.5 * ((sampled - mu) / std) ** 2
            log_prob = log_prob.sum(dim=1)
            loss = -(log_prob * advb).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        final_loss = epoch_loss / n_batches

    out_path = os.path.join(OUTPUT_DIR, f'{slug}.pt')
    torch.save(model.state_dict(), out_path)
    elapsed = time.time() - t_deck
    summary[slug] = {'snaps': len(snaps), 'baseline_R': baseline.item(), 'final_loss': final_loss, 'elapsed_s': elapsed}
    print(f'  → {out_path} (baseline_R={baseline.item():.3f}, final_loss={final_loss:.4f}, {elapsed:.1f}s)')
    del X, R, X_np, R_np, model, train_ds, train_loader, optimizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f'\n=== TOTAL {time.time()-t_global:.0f}s ===')
print(f'\n出力ディレクトリ: {OUTPUT_DIR}/')
import json as _json
with open(os.path.join(OUTPUT_DIR, '_summary.json'), 'w') as f:
    _json.dump(summary, f, indent=2)

## 5. 完了報告

`MyDrive/onepiece_nn/nn_per_deck_oneturn/` 配下に 16 つの `.pt` が 出力された。
ローカル `db/nn_per_deck_oneturn/` へ download してください。